# MILO - Cell type composition analysis

Publication [Dann et al, 2021](https://www.nature.com/articles/s41587-021-01033-z)  
Marioni Lab [GitHub](https://github.com/MarioniLab/miloR)  

Theis lab python adaption with [pertpy](https://github.com/theislab/pertpy)

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 

            # miloR
            library(miloR), 
            library(ggbeeswarm), 
            
            # SingleCellExperiment
            library(SingleCellExperiment), 

            # Data 
            library(tidyverse), 
            library(data.table), 

            # Plotting 
            library(patchwork), 
            library(cowplot), 
            
            # Parallel 
            library(BiocParallel), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [6]:
# Configure reticulate 
use_condaenv(condaenv="p.3.10.16-FD20250213HSCT", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

python:         /nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT/bin/python
libpython:      /nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT/lib/libpython3.10.so
pythonhome:     /nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT:/nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT
version:        3.10.16 | packaged by conda-forge | (main, Apr  8 2025, 20:53:32) [GCC 13.3.0]
numpy:          /nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT/lib/python3.10/site-packages/numpy
numpy_version:  2.2.6

NOTE: Python version was forced by use_python() function

In [7]:
random_seed <- 42
set.seed(random_seed)

In [8]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [9]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

In [10]:
# Parallel 
BPPARAM <- SerialParam()
register(BPPARAM)

In [11]:
# Reticulate python modules
sc <- import("scanpy", as="sc", convert=FALSE)
ad <- import("anndata", as="ad", convert=FALSE)

# Get Seurat meta data

In [12]:
so <- readRDS("data/object/pp.rds")
meta <- so@meta.data

# Import Seurat object with SCVI latent space from AnnData

In [ ]:
milo_import <- function(file, meta, celltype_domain) {

    adata <- sc$read_h5ad(file)

    # Filter by celltype domain 
    adata = adata[adata$obs$celltype_domain==celltype_domain]$copy()

    # AnnData to SCE
    X <- py_to_r(adata$X) %>% as.matrix() %>% t()
    
    obs_names <- py_to_r(adata$obs_names$to_list())
    var_names <- py_to_r(adata$var_names$to_list())
    
    latent <- py_to_r(adata$obsm["latent"]) %>% as.data.frame()
    rownames(latent) <- obs_names

    meta <- meta[obs_names, ]

    sce <- SingleCellExperiment::SingleCellExperiment(assays=list(countst=X), colData=meta, reducedDims=list(latent=as.matrix(latent)))
    
    # Milo from SCE
    milo <- miloR::Milo(sce)

    # Do not store counts 
    assays(milo) <- list()

    return(milo)
    
}

In [ ]:
milo_0 <- milo_import("data/object/pp_0.h5ad", meta, celltype_domain="$Immune\\ cells$")
milo_1 <- milo_import("data/object/pp_1.h5ad", meta, celltype_domain="$Immune\\ cells$")
milo_2 <- milo_import("data/object/pp_1.h5ad", meta, celltype_domain="$Non-hematopoetic$")

# Run Milo

In [ ]:
k_range <- seq(10, 100, 10)

In [ ]:
cache <- TRUE

In [ ]:
if(!cache) {
    
    for (k in k_range) {
        
        # Run makeNhoods with refinement_scheme="graph" to avoid calcNhoodDistance and set fdr.weighting="graph-only" testNhoods
        # https://github.com/MarioniLab/miloR/issues/99
    
        # Graph 
        milo_0 <- buildGraph(milo_0, k=k, d=10, reduced.dim="latent")
        milo_1 <- buildGraph(milo_1, k=k, d=10, reduced.dim="latent")
        milo_2 <- buildGraph(milo_2, k=k, d=10, reduced.dim="latent")

        milo_0 <- makeNhoods(milo_0, prop=0.2, k=k, d=10, refined=TRUE, reduced_dims="latent", refinement_scheme="graph")
        milo_1 <- makeNhoods(milo_1, prop=0.2, k=k, d=10, refined=TRUE, reduced_dims="latent", refinement_scheme="graph")
        milo_2 <- makeNhoods(milo_2, prop=0.2, k=k, d=10, refined=TRUE, reduced_dims="latent", refinement_scheme="graph")

        milo_0 <- countCells(milo_0, meta.data=data.frame(colData(milo_0)), sample="sample_name")
        milo_1 <- countCells(milo_1, meta.data=data.frame(colData(milo_1)), sample="sample_name")
        milo_2 <- countCells(milo_2, meta.data=data.frame(colData(milo_2)), sample="sample_name")
        
        saveRDS(milo_0, paste0("data/milo/milo_k", as.character(k), "_0.rds"), compress=TRUE)
        saveRDS(milo_1, paste0("data/milo/milo_k", as.character(k), "_1.rds"), compress=TRUE)
        saveRDS(milo_2, paste0("data/milo/milo_k", as.character(k), "_2.rds"), compress=TRUE)
    
    }

} else {

    milo_0 <- lapply(k_range, function(k) {readRDS(paste0("data/milo/milo_k", as.character(k), "_0.rds"))})
    milo_1 <- lapply(k_range, function(k) {readRDS(paste0("data/milo/milo_k", as.character(k), "_1.rds"))})
    milo_2 <- lapply(k_range, function(k) {readRDS(paste0("data/milo/milo_k", as.character(k), "_2.rds"))})

}

In [ ]:
names(milo_0) <- k_range
names(milo_1) <- k_range
names(milo_2) <- k_range

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)

hist_0 <- lapply(milo_0, function(i) {plotNhoodSizeHist(i) + ggtitle(paste("Blood\n k", i@.k)) + xlim(0, 300) + theme_global_set(1)})
hist_1 <- lapply(milo_1, function(i) {plotNhoodSizeHist(i) + ggtitle(paste("Skin hematopoetic\n k", i@.k)) + xlim(0, 300) + theme_global_set(1)})
hist_2 <- lapply(milo_2, function(i) {plotNhoodSizeHist(i) + ggtitle(paste("Skin non-hematopoetic\n k", i@.k)) + xlim(0, 300) + theme_global_set(1)})

wrap_plots(hist_0, ncol=10, nrow=1) 
wrap_plots(hist_1, ncol=10, nrow=1) 
wrap_plots(hist_2, ncol=10, nrow=1)

# Test neighborhoods

In [ ]:
milo_glm <- function(milo) {
    
    # Subset meta data 
    meta <- colData(milo) %>% data.frame()
    
    # Design 
    design_df <- meta[,c ("sample_name", "time_point", "patient_id"), drop=FALSE]
    design_df <- distinct(design_df)
    rownames(design_df) <- design_df$sample_name
    
    # Model matrix
    design <- model.matrix(~0+time_point, data=design_df)
    
    result_1 <- miloR::testNhoods(x=milo, design=design, design.df=design_df, model.contrasts="time_pointTx-time_pointBaseline", fdr.weighting="graph-overlap", reduced.dim="latent")
    result_2 <- miloR::testNhoods(x=milo, design=design, design.df=design_df, model.contrasts="time_pointD14-time_pointBaseline", fdr.weighting="graph-overlap", reduced.dim="latent")
    result_3 <- miloR::testNhoods(x=milo, design=design, design.df=design_df, model.contrasts="time_pointD100-time_pointBaseline", fdr.weighting="graph-overlap", reduced.dim="latent")
    result_4 <- miloR::testNhoods(x=milo, design=design, design.df=design_df, model.contrasts="time_pointGVHD-time_pointBaseline", fdr.weighting="graph-overlap", reduced.dim="latent")
    
    # P-value hist
    pvalue_hist_1 <- ggplot(result_1, aes(PValue)) + geom_histogram(bins=50) + ggtitle(paste("Baseline vs Tx\n k", milo@.k)) + theme_global_set(2)
    pvalue_hist_2 <- ggplot(result_2, aes(PValue)) + geom_histogram(bins=50) + ggtitle(paste("Baseline vs D14\n k", milo@.k)) + theme_global_set(2)
    pvalue_hist_3 <- ggplot(result_3, aes(PValue)) + geom_histogram(bins=50) + ggtitle(paste("Baseline vs D100\n k", milo@.k)) + theme_global_set(2)
    pvalue_hist_4 <- ggplot(result_4, aes(PValue)) + geom_histogram(bins=50) + ggtitle(paste("Baseline vs GVHD\n k", milo@.k)) + theme_global_set(2)

    # Volcano plot 
    vp_1 <- ggplot(result_1, aes(logFC, -log10(SpatialFDR))) + geom_point() + geom_hline(yintercept=1) + ggtitle(paste("Baseline vs Tx\n k", milo@.k)) + theme_global_set(2)
    vp_2 <- ggplot(result_2, aes(logFC, -log10(SpatialFDR))) + geom_point() + geom_hline(yintercept=1) + ggtitle(paste("Baseline vs D14\n k", milo@.k)) + theme_global_set(2)
    vp_3 <- ggplot(result_3, aes(logFC, -log10(SpatialFDR))) + geom_point() + geom_hline(yintercept=1) + ggtitle(paste("Baseline vs D100\n k", milo@.k)) + theme_global_set(2)
    vp_4 <- ggplot(result_4, aes(logFC, -log10(SpatialFDR))) + geom_point() + geom_hline(yintercept=1) + ggtitle(paste("Baseline vs GVHD\n k", milo@.k)) + theme_global_set(2)
    
    # Annotate hoods
    result_1 <- annotateNhoods(milo, result_1, coldata_col="celltype_low")
    result_2 <- annotateNhoods(milo, result_2, coldata_col="celltype_low")
    result_3 <- annotateNhoods(milo, result_3, coldata_col="celltype_low")
    result_4 <- annotateNhoods(milo, result_4, coldata_col="celltype_low")
    
    # Factor names 
    result_1$celltype_low <- factor(result_1$celltype_low, levels=rev(names(color$celltype_low_r))) %>% droplevels()
    result_2$celltype_low <- factor(result_2$celltype_low, levels=rev(names(color$celltype_low_r))) %>% droplevels()
    result_3$celltype_low <- factor(result_3$celltype_low, levels=rev(names(color$celltype_low_r))) %>% droplevels()
    result_4$celltype_low <- factor(result_4$celltype_low, levels=rev(names(color$celltype_low_r))) %>% droplevels()
    
    # Beeswarm 
    dabp_1 <- suppressMessages(plotDAbeeswarm(result_1, group.by="celltype_low", alpha=1.0) + ylab("") + ggtitle(paste("Baseline vs Tx\n k", milo@.k)) + theme_global_set(2))
    dabp_2 <- suppressMessages(plotDAbeeswarm(result_2, group.by="celltype_low", alpha=1.0) + ylab("") + ggtitle(paste("Baseline vs D14\n k", milo@.k))  + theme_global_set(2))
    dabp_3 <- suppressMessages(plotDAbeeswarm(result_3, group.by="celltype_low", alpha=1.0) + ylab("") + ggtitle(paste("Baseline vs D100\n k", milo@.k))  + theme_global_set(2))
    dabp_4 <- suppressMessages(plotDAbeeswarm(result_4, group.by="celltype_low", alpha=1.0) + ylab("") + ggtitle(paste("Baseline vs GVHD\n k", milo@.k)) + theme_global_set(2))

    # Combine plots 
    res_p <- pvalue_hist_1 + pvalue_hist_2 + pvalue_hist_3 + pvalue_hist_4 + vp_1 + vp_2 + vp_3 + vp_4 + dabp_1 + dabp_2 + dabp_3 + dabp_4 + plot_layout(ncol=12)
    
    # Cobine result table
    res_da <- list(result_1, result_2, result_3, result_4)
    
    # # Combine all
    resl <- list(res_p=res_p, res_da=res_da)
    
    return(resl)
    
}

## Blood 

In [ ]:
cache <- TRUE

In [ ]:
if(!cache) {

    resl_0 <- lapply(seq_along(k_range), function(k) milo_glm(milo=milo_0[[k]]))
    saveRDS(resl_0, "data/milo/resl_0.rds", compress=TRUE)

} else {

    resl_0 <- readRDS("data/milo/resl_0.rds")
    
}

In [ ]:
options(repr.plot.width=40, repr.plot.height=30)

wrap_plots(sapply(resl_0, "[", 1), ncol=1, nrow=10)

## Skin hematopoietic 

In [ ]:
cache <- TRUE

In [ ]:
if(!cache) {

    resl_1 <- lapply(seq_along(k_range), function(k) milo_glm(milo=milo_1[[k]]))
    saveRDS(resl_1, "data/milo/resl_1.rds", compress=TRUE)

} else {

    resl_1 <- readRDS("data/milo/resl_1.rds")
    
}

In [ ]:
options(repr.plot.width=40, repr.plot.height=30)

p_1 <- sapply(resl_1, "[", 1)
wrap_plots(p_1[c(1, 2, 3, 5, 7, 8, 9, 10)], ncol=1, nrow=10)

## Skin non-hematopoietic

In [ ]:
cache <- TRUE

In [ ]:
if(!cache) {

    resl_2 <- lapply(seq_along(k_range), function(k) milo_glm(milo=milo_2[[k]]))
    saveRDS(resl_2, "data/milo/resl_2.rds", compress=TRUE)

} else {

    resl_2 <- readRDS("data/milo/resl_2.rds")
    
}

In [ ]:
options(repr.plot.width=40, repr.plot.height=30)

wrap_plots(sapply(resl_2, "[", 1), ncol=1, nrow=10)

# Session info

In [ ]:
sessionInfo()